In [26]:
!uv pip install ultralytics
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.6/112.6 GB disk)


In [27]:
!apt-get install zip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zip is already the newest version (3.0-12build2).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [28]:
!unzip '/content/drive/MyDrive/Datasets/archive.zip' -d '/content/dataset'

unzip:  cannot find or open /content/drive/MyDrive/Datasets/archive.zip, /content/drive/MyDrive/Datasets/archive.zip.zip or /content/drive/MyDrive/Datasets/archive.zip.ZIP.


In [29]:
import os
import random
import shutil

source = "/content/drive/MyDrive/Datasets/extracted_frames"
target = "/content/final_dataset"

classes = ["focused", "distracted"]

split_ratio = 0.8

for cls in classes:

    images = os.listdir(f"{source}/{cls}")
    random.shuffle(images)

    split_idx = int(len(images) * split_ratio)

    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    os.makedirs(f"{target}/train/{cls}", exist_ok=True)
    os.makedirs(f"{target}/val/{cls}", exist_ok=True)

    for img in train_imgs:
        shutil.copy(
            f"{source}/{cls}/{img}",
            f"{target}/train/{cls}/{img}"
        )

    for img in val_imgs:
        shutil.copy(
            f"{source}/{cls}/{img}",
            f"{target}/val/{cls}/{img}"
        )

print("Done")

Done


In [30]:
model = YOLO("yolo11n-cls.pt")
results = model.train(data="/content/final_dataset",
        epochs=20,
        imgsz=224,
        batch=16,
        device=0,
        workers=4
    )


Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/final_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per

In [31]:

model = YOLO("/content/runs/classify/train-2/weights/best.pt")
predictions = model('/content/final_dataset/test.jpg')

result = predictions[0]

class_id = result.probs.top1

class_name = result.names[class_id]

confidence = float(result.probs.top1conf)

print(f"Prediction: {class_name}")
print(f"Confidence: {confidence:.2%}")


image 1/1 /content/final_dataset/test.jpg: 224x224 focused 1.00, distracted 0.00, 4.2ms
Speed: 11.5ms preprocess, 4.2ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Prediction: focused
Confidence: 99.79%
